# Byte-Level BPE — Exercise

Companion to [Tokenization § The Character vs Byte Problem](https://tanulsingh.github.io/blog/tokenization#the-character-vs-byte-problem).

Character-level BPE has a problem at scale: "characters" in Unicode means hundreds of thousands of code points (Chinese characters, Arabic, emoji, math symbols). Your initial vocabulary would be massive *before* a single merge happens.

Byte-level BPE fixes this elegantly. Every text is, at the lowest level, a sequence of UTF-8 bytes — and there are exactly 256 possible bytes. So the initial vocabulary is always exactly 256 entries, regardless of language.

You'll modify your character-level BPE to operate on bytes instead. The algorithm is identical; only the representation changes.

**Pre-requisite:** complete `01_bpe_exercise.ipynb` first — most of this notebook reuses your earlier code.

## Setup

In [ ]:
# TODO: imports (Counter, etc.)
# TODO: paste in or import: pre_tokenize, get_pair_counts, merge_pair from notebook 01

## Aside — How UTF-8 encodes text

Before writing the byte-level version, look at how Python encodes text to bytes:

```python
'hello'.encode('utf-8')        # b'hello'  (5 bytes — one per ASCII character)
'नमस्ते'.encode('utf-8')        # 18 bytes (each Devanagari char = 3 bytes)
'🤖'.encode('utf-8')            # 4 bytes (emoji)
```

This is why Hindi text is more expensive than English in token-based pricing: more bytes per character means more tokens after BPE.

In [ ]:
# Try it yourself
for text in ['hello', 'नमस्ते', '猫', '🤖', 'def __init__']:
    bytes_ = text.encode('utf-8')
    print(f"{text!r:20s} → {len(bytes_):2d} bytes  {list(bytes_)}")

## TODO 1 — Byte-level pre-tokenize

Mirror your character-level `pre_tokenize` but operate on **bytes** instead of characters. Each "token" is now an integer in `[0, 256)`.

For the end-of-word marker, you can use a special byte ID like `256` (out of UTF-8's range, so it's safe). Or skip the marker entirely if you're not splitting on words — your call.

**Suggested signature:**

```python
def pre_tokenize_bytes(corpus: list[str]) -> dict[tuple[int, ...], int]:
    # convert each string to bytes, split on whitespace bytes if you want word-level grouping
    # return {byte_tuple: frequency}
```

**Tip:** you can keep things simple by NOT splitting on whitespace at the byte level. The blog's SentencePiece section makes the case that this is actually cleaner for non-English languages (which don't have whitespace separators). Try both ways.

In [ ]:
def pre_tokenize_bytes(corpus: list[str]) -> dict[tuple[int, ...], int]:
    """Convert a corpus to byte-level word frequencies."""
    # TODO 1: implement — convert each text to bytes, then to a tuple of ints
    pass

In [ ]:
# Sanity check
freqs = pre_tokenize_bytes(["hi", "hi there", "नमस्ते"])
for word, freq in freqs.items():
    print(word, '→', freq)
# You should see ints in [0, 256) for all the chars.
# Hindi entries will have higher byte values (>= 224) due to UTF-8 encoding

## TODO 2 — Reuse `get_pair_counts` and `merge_pair`

Good news: `get_pair_counts` and `merge_pair` from notebook 01 work as-is on tuples of any type. Tuples of ints work the same as tuples of strings.

The only thing that changes: when you merge byte 116 (`'t'`) and byte 104 (`'h'`), the result is a *new* token with a new ID, not the string `'th'`. Pick a convention: I'd suggest assigning new IDs starting at 256 and going up.

**Suggested signature:**

```python
class ByteLevelBPETokenizer:
    def __init__(self):
        self.merges: list[tuple[int, int]] = []     # learned (a_id, b_id) merges
        self.vocab: dict[int, bytes] = {i: bytes([i]) for i in range(256)}   # id → bytes
        self.next_id = 256
```

When you merge `(116, 104)`, you'd record `116, 104 → 256` in `self.merges` (and add `vocab[256] = b'th'`). The next merge gets ID 257, and so on.

In [ ]:
class ByteLevelBPETokenizer:
    def __init__(self):
        self.merges: list[tuple[int, int]] = []
        self.vocab: dict[int, bytes] = {i: bytes([i]) for i in range(256)}
        self.next_id = 256

    def train(self, corpus: list[str], vocab_size: int):
        """Train byte-level BPE on a corpus."""
        # TODO 2a: word_freqs = pre_tokenize_bytes(corpus)
        # TODO 2b: while len(self.vocab) < vocab_size:
        #             pair_counts = get_pair_counts(word_freqs)
        #             if not pair_counts: break
        #             best_pair = max(pair_counts, key=pair_counts.get)
        #             new_id = self.next_id
        #             self.next_id += 1
        #             self.vocab[new_id] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]
        #             self.merges.append(best_pair)
        #             # Apply the merge by replacing (a_id, b_id) with new_id
        #             # NOTE: merge_pair from notebook 01 does string concat — for ints, you need a slight tweak.
        #             #       Either: rewrite merge_pair to take a `replacement` argument, or write merge_pair_int locally.
        pass

    def encode(self, text: str) -> list[int]:
        """Encode text as a list of token IDs."""
        # TODO 2c: convert text to bytes, then replay self.merges in order on each word
        # Return a flat list of int IDs
        pass

## TODO 3 — Train and observe multilingual behavior

Train on a small mixed-language corpus and inspect:

1. The vocab — what got merged?
2. Token counts for the same meaning in different languages
3. Which language gets the best compression?

In [ ]:
# TODO 3:
#   - Build a small corpus with English, Hindi, and Japanese sentences
#   - Train ByteLevelBPETokenizer with vocab_size=300 (only ~44 merges past the 256 base)
#   - Encode the same meaning in all three languages — compare token counts
#   - Observation: which language gets shorter encodings? Why?

## Run the tests

In [ ]:
from tests import run_byte_level_bpe_tests
# run_byte_level_bpe_tests(ByteLevelBPETokenizer)